# Caso E · 04 Predicción de generación solar

> _Tutorial · Caso de uso: **E — Meteo & solar** · Capa Medallion: **oro** · Spec: `docs/specs/synthetic-bms/02-domain-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Entrenar un regressor para `solar_irradiance` con T, hora del día y día del año. Punto de partida para predicción FV.


## 2. Qué se aprende

- Modelo simple de GHI con features físicas.
- Métrica RMSE (W/m²).
- Por qué la predicción solar a 24h es viable con buenos features.


## 3. Contexto del caso de uso

Predicción solar es tool del chatbot.


## 4. Relación con CENTINELA+

Sirve a Caso B y Caso H.


## 5. Relación con Medallion

Oro: modelo entrenado.


## 6. Datos de entrada

Oro features Caso E.


## 7. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [1]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


ROOT=C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS, SEED=42, default_bucket=telemetry


## 8. Schema CAPTIA esperado

No aplica.


## 9. Carga de datos o mock

Cargamos.


In [2]:
parquet_path = ROOT / "output" / "case_E" / "weather_features.parquet"
if parquet_path.exists():
    df = pd.read_parquet(parquet_path)
else:
    df, _ = mocks.make_era5_xativa_mock()
    df = df.set_index("timestamp")

X = pd.DataFrame(index=df.index)
X["hour"] = df.index.hour
X["doy"] = df.index.dayofyear
X["t"] = df["t_air_c"]
X["hour_sin"] = np.sin(2 * np.pi * X["hour"] / 24)
X["hour_cos"] = np.cos(2 * np.pi * X["hour"] / 24)
X["y"] = df["ghi_w_m2"]
X = X.dropna()
print(X.shape)


(720, 6)


## 10. Exploración paso a paso

Split temporal.


In [3]:
n = len(X)
i = int(n * 0.7)
X_tr, X_te = X.iloc[:i], X.iloc[i:]
y_tr, y_te = X_tr.pop("y"), X_te.pop("y")


## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

Modelo y métrica.


In [4]:
from sklearn.ensemble import RandomForestRegressor

m = RandomForestRegressor(n_estimators=200, random_state=SEED).fit(X_tr, y_tr)
y_pred = m.predict(X_te)
rmse = float(np.sqrt(np.mean((y_te.values - y_pred) ** 2)))
print({"RMSE_W_m2": round(rmse, 2)})


{'RMSE_W_m2': 140.04}


## 13. Visualizaciones explicativas

Curva real vs predicción 7 días.


In [5]:
plt.figure(figsize=(10, 3))
plt.plot(y_te.index[:24*7], y_te.values[:24*7], label="real", color="#FF5722")
plt.plot(y_te.index[:24*7], y_pred[:24*7], label="modelo", color="#3F51B5")
plt.legend(); plt.title("GHI predicho 7 días")
plt.tight_layout()


## 14. Validaciones

RMSE < 200 W/m² aceptable para clase.


In [6]:
assert rmse < 250


## 15. Errores comunes

1. **Olvidar ciclo anual** (dayofyear).
2. **Predecir GHI nocturno** distinto de cero.
3. **No clip a 0**.


## 16. Ejercicios propuestos

1. Añade `cloud_cover` (mock razonable).
2. Convierte GHI a Wh/día y compara con energía esperada FV.
3. Ensaya XGBoost y compara.


## 17. Cómo se reutiliza con datos reales

Mismo modelo, datos de ERA5 real o AEMET. Para FV añadir factor de rendimiento del panel + temperatura.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `06_case_F_mlops/01_mlflow_lakefs_overview.ipynb`.
- Documento web del caso: `docs/use-cases/case-e-weather-solar.md`.


## 19. Marco teórico (nivel doctoral)

### Modelo de irradiancia solar global (Iqbal 1983)

$$
G_h(t) = G_b(t) + G_d(t), \quad G_b(t) = G_{sc} \cdot \tau_b(t) \cdot \cos\theta_z(t)
$$

con $G_{sc} = 1361$ W/m² constante solar y $\theta_z$ ángulo cenital del sol:

$$
\cos\theta_z = \sin\delta \sin\phi + \cos\delta \cos\phi \cos\omega
$$

donde $\delta$ es declinación solar, $\phi$ latitud (Xátiva 38.99°N) y
$\omega$ ángulo horario.

### Clear-sky index

$$
k_c(t) = \frac{G_h(t)}{G_{clear}(t)} \in [0, 1]
$$

separa astronomía (determinista) de meteorología (estocástica).

### Predictor XGBoost para FV

$$
\hat{P}(t+h) = P_{nominal} \cdot \eta_{panel} \cdot \text{XGB}(k_c(t), T_{out}, t_{hora}, t_{año})
$$

### Métrica Skill Score

$$
\text{Skill} = 1 - \frac{\text{RMSE}_{model}}{\text{RMSE}_{persistence}}
$$

Objetivo Simarro: $\text{nMAE} \leq 8\%$ a 24 h, $\text{Skill} \geq 0.3$.


## 20. Visión corporativa CAPTIA

### Propuesta de valor

Predicción solar permite optimizar despacho energético en centros con FV instalada y planificar climatización aprovechando picos de radiación. CAPTIA puede ofrecer este caso como **producto añadido** a centros con paneles.

### ROI estimado

| Concepto | Valor |
|---|---|
| Optimización despacho FV (centro 50 kWp) | +800 €/año |
| Sinergia con Caso B forecast | +500 €/año |
| Coste integración ERA5+AEMET | -1 200 € one-time |
| **Payback** | **~12 meses** |


## 21. Bibliografía y referencias

- Iqbal, M. (1983). *An Introduction to Solar Radiation*. Academic Press.
- ECMWF (2024). *ERA5 Reanalysis Documentation*. Copernicus Climate Change Service.
- AEMET. *Open Data Portal*. https://opendata.aemet.es
- Holmgren, W. F. et al. (2018). *pvlib python: a python package for modeling solar energy systems*. JOSS 3(29).
